# Voice Coding Coach — Phase 2 (`T-03`) & `R1–R9` Reproducible Verification & Promotion Workflow (`PHASE_2_COLAB.ipynb`)

**Capabilities & Execution Modes:**
1. **Drive-ID Resolution & Materialization (Colab / Google API Mode):** Resolves the canonical implementation root (`1mmsMOQQ-mbRhmEUawG-AnQs_4NwQZYy5`) and continuation root (`180tEut_BJOupH4fFUdQMlQ07u1q6a_wq` / `repaired_v2_p2` `1zexzpbq9AGfuIKzthacD1YqEPar4tlyt`) by Drive ID via `google.auth.default()`, downloads the baseline, applies the explicit file-ID-keyed `OVERLAY_MANIFEST` with SHA-256 verification, and installs declared dependencies in temporary storage.
2. **Local / Offline Checkout Mode:** When `VCC_LOCAL_REPO_ROOT` is set (or when running outside Google Colab against an existing local checkout), stages an isolated working copy in temporary storage without requiring Google Colab.
3. **Read-Only Drift & Compile Gates First:** Executes `compileall`, `runner_image_isolated_import`, `export_openapi.py --check`, and `check_schema_drift.py --check --self-test` **before** any regeneration.
4. **Full Backend, Worker & Frontend Suites:** Runs `backend/tests`, `exec_worker/tests`, `test_r1_to_r9_regressions.py`, `test_speech_foundation.py`, `check_secrets.py`, `smoke_gnani.py --allow-blocked`, and `npm run typecheck`, `npm test`, `npm run build`.
5. **Separate External Gates & Verified Read-Back:** Keeps live Gnani (`LIVE_EXTERNAL_PROVIDER`) and live gVisor (`LIVE_GVISOR_ISOLATION`) gates visibly separate (never reporting skipped external gates as `PASS`), persists partial logs after every command, and fails with `RuntimeError` if any required check fails.

## Review and safety boundary
Every copy/download is quarantined data until `VCC_REVIEWED_MANIFEST_FILE` outside staged source matches every byte after independent manual review. Format: `{"reviewed": true, "files": {"relative/path": "lowercase SHA-256"}}`. Self-generated hashes from unreviewed downloads are not approval. Existing reviewed dependencies are preferred; explicit opt-in installation (`VCC_INSTALL_REVIEWED_DEPENDENCIES=1`) requires hashed wheel-only requirements and npm locks with lifecycle scripts disabled. Missing review/locks/hashes block. No archives are extracted or run.
Runner packaging is import-only, not trace/isolation evidence. Required missing/zero-test/failed gates and blocked gVisor acceptance prevent promotion. Colab without genuine runsc stays BLOCKED_EXTERNAL; never invoke learner source locally or replace evaluation with mocks. Live Gnani requires explicit `VCC_RUN_LIVE_GNANI=1`. Redacted full output is retained outside source. This notebook has not itself proved external execution or Drive source reconciliation.


In [ ]:
# Cell 1: Initialize Workspace, Optional Colab Auth / Secrets, and Drive Root IDs
import hashlib
import io
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tarfile
import tempfile
import time
from datetime import datetime, timezone

CANONICAL_IMPL_FOLDER_ID = '1mmsMOQQ-mbRhmEUawG-AnQs_4NwQZYy5'
CONTINUATION_FOLDER_ID = '180tEut_BJOupH4fFUdQMlQ07u1q6a_wq'
REPAIRED_P2_FOLDER_ID = '1zexzpbq9AGfuIKzthacD1YqEPar4tlyt'

IN_COLAB = False
try:
    from google.colab import auth as colab_auth, drive, userdata  # type: ignore
    IN_COLAB = True
    drive.mount('/content/drive')
    colab_auth.authenticate_user()
except ImportError:
    print('Running in local/offline non-Colab environment.')

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_p2_verified'
WORKSPACE_ROOT = Path(tempfile.mkdtemp(prefix='vcc_review_workspace_'))
STAGED_REPO = WORKSPACE_ROOT / 'implementation'
ARTIFACTS_OUT = WORKSPACE_ROOT / 'artifacts'
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACTS_OUT.mkdir(parents=True, exist_ok=True)
print('Initialized temporary workspace:', WORKSPACE_ROOT)

TRUSTED_TOOLS = WORKSPACE_ROOT / 'trusted_tools'
TRUSTED_TOOLS.mkdir()


In [ ]:
# Cell 2: Non-Recursive Archive Helper, Sentinel Self-Test (R6), and SHA-256 Read-Back Verifier
EXCLUDED_DIR_NAMES = {
    '.git', 'node_modules', '__pycache__', '.pytest_cache', '.mypy_cache',
    '.ruff_cache', '.venv', 'venv', 'dist', 'build', 'artifacts', '.tts_cache'
}
EXCLUDED_SUFFIXES = {'.pyc', '.pyo', '.sqlite3', '.db', '.pem', '.key', '.tar', '.gz', '.zip', '.bz2', '.xz', '.7z'}

def is_approved_source_file(path: Path, root: Path) -> bool:
    if path.is_symlink() or not path.is_file():
        return False
    rel = path.relative_to(root)
    if root.is_symlink() or any((root / parent).is_symlink() for parent in rel.parents):
        return False
    if any(part in EXCLUDED_DIR_NAMES or part.startswith('.venv') for part in rel.parts):
        return False
    if path.name.startswith('.env') or path.suffix in EXCLUDED_SUFFIXES:
        return False
    return True

def build_verified_source_archive(src_root: Path, out_tar_gz: Path) -> dict[str, str]:
    manifest: dict[str, str] = {}
    with tarfile.open(out_tar_gz, 'w:gz') as tar:
        for p in sorted(src_root.rglob('*')):
            if not is_approved_source_file(p, src_root):
                continue
            rel_str = p.relative_to(src_root).as_posix()
            tar.add(p, arcname=rel_str, recursive=False)
            manifest[rel_str] = hashlib.sha256(p.read_bytes()).hexdigest()
    return manifest

def verify_archive_sentinel_self_test() -> None:
    with tempfile.TemporaryDirectory() as tmp:
        tmp_root = Path(tmp) / 'repo'
        (tmp_root / 'backend' / 'app').mkdir(parents=True)
        (tmp_root / 'backend' / 'app' / 'main.py').write_text('print(1)\n', encoding='utf-8')
        (tmp_root / 'nested' / 'node_modules' / 'pkg').mkdir(parents=True)
        (tmp_root / 'nested' / 'node_modules' / 'pkg' / 'index.js').write_text('x', encoding='utf-8')
        (tmp_root / 'nested' / '.env').write_text('SECRET=1\n', encoding='utf-8')
        tar_path = Path(tmp) / 'test.tar.gz'
        build_verified_source_archive(tmp_root, tar_path)
        with tarfile.open(tar_path, 'r:gz') as tar:
            names = tar.getnames()
        assert names == ['backend/app/main.py'], f'Sentinel leak detected: {names}'
    print('PASS: Archive exclusion sentinel self-test verified (recursive=False).')

def copy_with_sha256_readback(src: Path, dst: Path) -> str:
    src_bytes = src.read_bytes()
    expected_sha = hashlib.sha256(src_bytes).hexdigest()
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        backup = dst.with_name(f'{dst.name}.prev_{int(time.time())}')
        shutil.copy2(dst, backup)
    dst.write_bytes(src_bytes)
    actual_sha = hashlib.sha256(dst.read_bytes()).hexdigest()
    if actual_sha != expected_sha:
        raise RuntimeError(f'SHA-256 read-back mismatch for {dst}: {actual_sha} != {expected_sha}')
    return expected_sha

verify_archive_sentinel_self_test()

# Review is external and byte-exact, never inferred from a downloaded bundle's own hashes.
from pathlib import PurePosixPath
import re

def safe_relative_path(value: str) -> Path:
    path = PurePosixPath(value)
    if not value or path.is_absolute() or '..' in path.parts or '\\' in value or ':' in value:
        raise ValueError('Unsafe remote/manifest path')
    if path.as_posix() != value or any(ord(char) < 32 for char in value):
        raise ValueError('Noncanonical remote/manifest path')
    return Path(*path.parts)

def verify_reviewed_tree(root: Path) -> None:
    setting = os.environ.get('VCC_REVIEWED_MANIFEST_FILE', '').strip()
    if not setting:
        raise RuntimeError('BLOCKED_EXTERNAL: Supply an independently reviewed SHA-256 manifest before install/import/run.')
    manifest_path = Path(setting).resolve()
    if manifest_path.is_relative_to(root.resolve()):
        raise RuntimeError('BLOCKED_EXTERNAL: Manifest must be supplied outside staged source.')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    expected = manifest.get('files')
    if manifest.get('reviewed') is not True or not isinstance(expected, dict) or not expected:
        raise RuntimeError('BLOCKED_EXTERNAL: Expected reviewed=true and a nonempty files mapping after manual review.')
    for name, digest in expected.items():
        safe_relative_path(name)
        if not isinstance(digest, str) or re.fullmatch('[0-9a-f]{64}', digest) is None:
            raise ValueError('Invalid reviewed SHA-256 digest')
    paths = list(root.rglob('*'))
    if any(path.is_symlink() for path in paths):
        raise RuntimeError('BLOCKED_EXTERNAL: Symlinks forbidden in staged source.')
    actual = {p.relative_to(root).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in paths if p.is_file()}
    if actual != expected:
        raise RuntimeError('BLOCKED_EXTERNAL: Staged tree differs from independently reviewed manifest; no code executed.')
    print('PASS: External reviewed manifest matches every staged file; hashes are not sandbox acceptance.')


In [ ]:
# Cell 3: Materialize Canonical Baseline + File-ID-Keyed Repaired P2 Overlay
REPAIRED_P2_OVERLAY_MAP = [
    {'file_id': '1oOnRNJMH_3WmDHjMLx0aknnXPW9M22OQ', 'target': 'backend/app/models/enums.py'},
    {'file_id': '1lH0UEEvuOFgb5LBDUgYeBf_VUgABZ3me', 'target': 'backend/app/models/domain.py'},
    {'file_id': '1fr-V631ChELnJsi_fg2DWQc3jWThEHwe', 'target': 'backend/app/models/api_schemas.py'},
    {'file_id': '1rglv-HbvIRCD4yeHfqhr6bw0vOGGP2Lw', 'target': 'backend/app/models/__init__.py'},
    {'file_id': '1SMZc0u8iB_BRZv88dWIt9pMsbw4zVzsq', 'target': 'backend/scripts/export_openapi.py'},
    {'file_id': '1a4WPZ7u2odhQsgwfwgb2W3WkA1Rjkvrb', 'target': 'backend/openapi.json'},
    {'file_id': '1gCsl_Qh4kOtjuyd7uhvFmDzCEO2qRMRA', 'target': 'scripts/check_schema_drift.py'},
    {'file_id': '16aaJuaC8eE_U5bkdYFdRErm36Pm0boFT', 'target': 'frontend/src/types/api.ts'},
    {'file_id': '1n6QDPcai8oAD1nP1O1gQJ8_lVGpF-ANt', 'target': 'backend/tests/unit/test_contracts.py'},
    {'file_id': '1eqBBzBj05MmfDiYIodHHS_73W8YkboTU', 'target': 'exec_worker/runner/safe_serializer.py'},
    {'file_id': '1pjUakObDD5KnEla__RfcHfsUXQscTHu3', 'target': 'exec_worker/runner/in_sandbox_tracer.py'},
    {'file_id': '1GoGH7s28pXKoIIauBAXNM_Gl-YwH3_BW', 'target': 'exec_worker/runner/run_invocation.py'},
    {'file_id': '1d26eUKKl0TWbBBeENnTn5vSw2OaQlN_Z', 'target': 'exec_worker/Dockerfile.runner'},
    {'file_id': '1gBQr5YU5UMxzsEyusR2iHq1uGQlmWJwd', 'target': 'exec_worker/worker_app/trace_recorder.py'},
    {'file_id': '1EOZcN7Dgs3NhkZIyBs1XH6lKM6QojH5Y', 'target': 'exec_worker/worker_app/sandbox_runner.py'},
    {'file_id': '1SP6lWpVNMReSuBOnIoKfbX6_xk5wDTiK', 'target': 'exec_worker/worker_app/verifier_harness.py'},
    {'file_id': '1u6nKAGzW7PLdOmoJY0sT8PSpr6UHm_Ot', 'target': 'exec_worker/tests/test_sandbox_isolation.py'},
    {'file_id': '1UHNvvEUy6cvIOF2Ydlgqne9a6IjugaYX', 'target': 'backend/tests/unit/test_r1_to_r9_regressions.py'},
    {'file_id': '14FR2-PQqtFgwxOBdh4m7h68PE30DdXMa', 'target': 'backend/app/speech/__init__.py'},
    {'file_id': '19kwtVsC4dE7RWUfLG25NH4dck8X3eMDZ', 'target': 'backend/app/speech/audio_utils.py'},
    {'file_id': '1tJV6jN7IV-eK9r_0XOqR0zmlrvLb5NfH', 'target': 'backend/app/speech/ast_keyterms.py'},
    {'file_id': '1aJWreWZy83JV-oXL2sBMwCUlZv8VfETM', 'target': 'backend/app/speech/pre_tts_normalizer.py'},
    {'file_id': '1gr2BDLrxhFvmxReChft9KKvcl4B0o_Al', 'target': 'backend/app/speech/transcript_normalizer.py'},
    {'file_id': '1ZI_PXtDHPYodNR68KpCYpwk9FCqz88Nb', 'target': 'backend/app/speech/tts_cache.py'},
    {'file_id': '19yGGjXMAjMyKjirWvLmkNbmhI9rLImQH', 'target': 'backend/app/speech/gnani_adapter.py'},
    {'file_id': '1VDqQ6PDHq8oWSIJQ1Gakyyl0A97c07Lz', 'target': 'backend/app/speech/gatekeeper.py'},
    {'file_id': '1wdUyGJBmCxdbVewcSs6rtk-GQ_0UTZ3v', 'target': 'scripts/smoke_gnani.py'},
    {'file_id': '1HARzQEKDz4gdh9wh2gBLjVj3_f1nkaeM', 'target': 'backend/tests/unit/test_speech_foundation.py'}
]

OVERLAY_MANIFEST = REPAIRED_P2_OVERLAY_MAP

def stage_repository() -> Path:
    local_override = os.environ.get('VCC_LOCAL_REPO_ROOT', '').strip()
    candidate_locals = [
        Path(local_override) if local_override else None,
        Path.cwd(),
        Path.cwd().parent,
    ]
    for cand in candidate_locals:
        if cand is not None and (cand / 'backend' / 'app' / 'main.py').is_file():
            STAGED_REPO.mkdir(exist_ok=False)
            for source in sorted(cand.resolve().rglob('*')):
                if source.is_relative_to(WORKSPACE_ROOT):
                    continue
                if is_approved_source_file(source, cand.resolve()):
                    destination = STAGED_REPO / safe_relative_path(source.relative_to(cand.resolve()).as_posix())
                    copy_with_sha256_readback(source, destination)
            print('Staged local repository checkout into:', STAGED_REPO)
            return STAGED_REPO

    if IN_COLAB:
        import google.auth
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload

        creds, _ = google.auth.default()
        service = build('drive', 'v3', credentials=creds)

        def _download_file(fid: str, dest: Path) -> None:
            dest.parent.mkdir(parents=True, exist_ok=True)
            req = service.files().get_media(fileId=fid)
            buf = io.BytesIO()
            downloader = MediaIoBaseDownload(buf, req)
            done = False
            while not done:
                _, done = downloader.next_chunk()
            dest.write_bytes(buf.getvalue())

        def _download_folder_recursive(folder_id: str, dest_dir: Path) -> None:
            dest_dir.mkdir(parents=True, exist_ok=True)
            page_token = None
            items = []
            while True:
                res = service.files().list(
                    q=f"'{folder_id}' in parents and trashed = false",
                    fields='nextPageToken, files(id, name, mimeType, modifiedTime)',
                    pageToken=page_token,
                ).execute()
                items.extend(res.get('files', []))
                page_token = res.get('nextPageToken')
                if not page_token:
                    break
            seen = set()
            for item in items:
                relative = safe_relative_path(item['name'])
                if len(relative.parts) != 1 or item['name'] in seen:
                    raise ValueError('Ambiguous or unsafe Drive filename; exact overlay IDs required')
                seen.add(item['name'])
                if item['name'] in EXCLUDED_DIR_NAMES or item['name'].startswith(('.env', '.venv')) or relative.suffix in EXCLUDED_SUFFIXES:
                    continue
                mime = item['mimeType']
                if mime == 'application/vnd.google-apps.folder':
                    _download_folder_recursive(item['id'], dest_dir / item['name'])
                elif not mime.startswith('application/vnd.google-apps.'):
                    _download_file(item['id'], dest_dir / item['name'])

        STAGED_REPO.mkdir(exist_ok=False)
        _download_folder_recursive(CANONICAL_IMPL_FOLDER_ID, STAGED_REPO)
        for entry in OVERLAY_MANIFEST:
            _download_file(entry['file_id'], STAGED_REPO / safe_relative_path(entry['target']))
        print('Materialized canonical baseline + repaired P2 overlay from Drive IDs into:', STAGED_REPO)
        return STAGED_REPO

    raise RuntimeError('BLOCKED_EXTERNAL: No checkout staged; set VCC_LOCAL_REPO_ROOT.')

TARGET_REPO = stage_repository()


In [ ]:
verify_reviewed_tree(TARGET_REPO)
RUNNER = TARGET_REPO / 'scripts' / 'run_phase2_checks.py'
if not RUNNER.is_file():
    raise RuntimeError('Required runner missing; no checks passed.')
child_env = {key: value for key, value in os.environ.items() if not re.search('KEY|TOKEN|SECRET|PASSWORD|CREDENTIAL', key, re.I)}
child_env.update({'CI': 'true', 'VCC_ALLOW_PUBLIC_CUSTOM_CODE': 'false', 'VCC_RUN_LIVE_GVISOR_ACCEPTANCE': '0'})
command = [sys.executable, '-I', str(RUNNER), '--repo-root', str(TARGET_REPO), '--output-dir', str(ARTIFACTS_OUT)]
# Reviewed runner only; notebook content below is source text, not a local learner invocation.
# Colab without genuine gVisor remains BLOCKED_EXTERNAL.
# No live provider calls unless explicitly requested. Never echo credential values.
LIVE_GNANI = os.environ.get('VCC_RUN_LIVE_GNANI', '') == '1'
if LIVE_GNANI:
    provider_key = os.environ.get('VCC_GNANI_API_KEY', '').strip()
    if IN_COLAB and not provider_key:
        try:
            provider_key = userdata.get('VCC_GNANI_API_KEY') or ''
        except Exception:
            raise RuntimeError('BLOCKED_EXTERNAL: Requested provider secret unavailable.') from None
    if provider_key:
        child_env['VCC_GNANI_API_KEY'] = provider_key
    command.append('--run-live-gnani')
# Explicit opt-in only, after review. Missing full hashes/locks must fail closed.
if os.environ.get('VCC_INSTALL_REVIEWED_DEPENDENCIES', '') == '1':
    subprocess.run([sys.executable, '-I', '-m', 'pip', 'install', '--require-hashes', '--only-binary=:all:', '-r', str(TARGET_REPO / 'backend' / 'requirements.txt')], cwd=str(TRUSTED_TOOLS), env=child_env, check=True)
    subprocess.run(['npm', 'ci', '--ignore-scripts', '--no-audit', '--no-fund', '--prefix', str(TARGET_REPO / 'frontend')], cwd=str(TRUSTED_TOOLS), env=child_env, check=True)
process = subprocess.run(command, cwd=str(TRUSTED_TOOLS), env=child_env, capture_output=True, text=True, check=False, timeout=1800)
runner_output = process.stdout + '\n' + process.stderr
for key, value in {**os.environ, **child_env}.items():
    if value and re.search('KEY|TOKEN|SECRET|PASSWORD|CREDENTIAL', key, re.I):
        runner_output = runner_output.replace(value, '[REDACTED]')
(ARTIFACTS_OUT / 'runner_output.txt').write_text(runner_output, encoding='utf-8')
print(runner_output)
if process.returncode != 0:
    raise RuntimeError('Required verification failed or BLOCKED_EXTERNAL; no promotion.')
